In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\Alkaram computers\Desktop\LIMEOX\ASSESSMENT # 7\News_Classification_Dataset.csv")

TEXT_COL = "Title"
LABEL_COL = "Category"

df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
df.columns = ["headline", "original_label"]

print(df.shape)
print(df["original_label"].value_counts())

(4999, 2)
original_label
Business      1896
Markets       1518
Politics       862
Technology     452
Health         144
Energy         127
Name: count, dtype: int64


In [19]:
from sentence_transformers import SentenceTransformer
import time

model = SentenceTransformer("all-MiniLM-L6-v2")

label_descriptions = {
    "Business": "business news about companies, corporate strategy, deals, management, commerce",
    "Markets": "stock market news about trading, indices, investors, IPOs, shares, forex",
    "Politics": "political news about government, elections, policy, legislation, geopolitics",
    "Technology": "technology news about software, AI, gadgets, internet, tech products, startups",
    "Health": "health news about medicine, disease, hospitals, vaccines, pharma, wellness",
    "Energy": "energy news about oil, gas, renewables, power grids, electricity, nuclear"
}

labels = list(label_descriptions.keys())
label_texts = list(label_descriptions.values())

start = time.time()

label_embeddings = model.encode(label_texts, convert_to_numpy=True, normalize_embeddings=True)
headline_embeddings = model.encode(
    df["headline"].astype(str).tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True
)

sims = headline_embeddings @ label_embeddings.T
best_idx = sims.argmax(axis=1)
best_score = sims.max(axis=1)

df["pred_label"] = [labels[i] for i in best_idx]
df["pred_conf"] = best_score

print(f"Done in {time.time() - start:.1f}s")
print(df["pred_label"].value_counts())

Batches: 100%|██████████| 79/79 [00:55<00:00,  1.43it/s]

Done in 55.8s
pred_label
Markets       1668
Technology    1096
Business       865
Energy         739
Health         326
Politics       305
Name: count, dtype: int64


In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_all = le.fit_transform(df["original_label"].values)
X_all = headline_embeddings

print(le.classes_)
print(X_all.shape, y_all.shape)

['Business' 'Energy' 'Health' 'Markets' 'Politics' 'Technology']
(4999, 384) (4999,)


In [21]:
from cleanlab.filter import find_label_issues
from sklearn.model_selection import cross_val_predict
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
pred_probs = cross_val_predict(lda, X_all, y_all, cv=5, method="predict_proba")

issue_idx = find_label_issues(
    labels=y_all,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
)

print(f"Found {len(issue_idx)} likely label issues out of {len(y_all)} ({len(issue_idx)/len(y_all)*100:.1f}%)")

issues_df = df.iloc[issue_idx].copy()
issues_df["suggested_label"] = le.inverse_transform(pred_probs[issue_idx].argmax(axis=1))
issues_df["confidence"] = pred_probs[issue_idx].max(axis=1)

print(issues_df[["headline", "original_label", "suggested_label", "confidence"]].head(20))

Found 1556 likely label issues out of 4999 (31.1%)
                                               headline original_label  \
3937  Australia's tropical trees emit more carbon th...         Health   
3615  Last year saw biggest increase of CO2 in atmos...         Health   
1804  Next-Generation Targeted Therapies Reshape Pre...       Business   
3103  If you were notified by City of Hope regarding...         Health   
1879  Novo Nordisk, Omeros Sign Asset Purchase, Lice...         Health   
1711  Vistagen to Present at The Menopause Society 2...       Business   
3480  Carbon dioxide levels hit record last year, ca...         Health   
3213         NGOs report possible forced labor on ships         Health   
4535  Skyharbour Partner Company Terra Clean Energy ...       Politics   
2681  WESTERN MIDSTREAM COMPLETES ACQUISITION OF ARI...         Health   
4421  Liquidia Insider Sold Shares Worth $272,258, A...         Health   
2526  Scanogen Receives FDA Breakthrough Device Desi...     T

In [22]:
CONF_THRESHOLD = 0.6

df["final_label"] = df["original_label"]
high_conf_fixes = issues_df[issues_df["confidence"] > CONF_THRESHOLD]
df.loc[high_conf_fixes.index, "final_label"] = high_conf_fixes["suggested_label"]

print(f"Fixed {len(high_conf_fixes)} labels out of {len(issues_df)} flagged issues (threshold={CONF_THRESHOLD})")
print(df["final_label"].value_counts())

changed = df[df["original_label"] != df["final_label"]]
print(changed.groupby(["original_label", "final_label"]).size().sort_values(ascending=False).head(15))

Fixed 1357 labels out of 1556 flagged issues (threshold=0.6)
final_label
Business      1974
Markets       1417
Politics       818
Technology     447
Health         174
Energy         169
Name: count, dtype: int64
original_label  final_label
Markets         Business       344
Business        Markets        269
Politics        Business       107
Technology      Business        98
Business        Technology      97
                Politics        70
Politics        Markets         45
Business        Energy          40
Markets         Politics        40
Business        Health          38
Energy          Business        25
Markets         Technology      24
                Energy          20
Health          Business        18
Technology      Politics        18
dtype: int64


In [23]:
from sklearn.model_selection import train_test_split

y_clean = le.fit_transform(df["final_label"].values)
X_train, X_test, y_train, y_test = train_test_split(
    headline_embeddings, y_clean, test_size=0.2, stratify=y_clean, random_state=42
)

print("Train dist after cleaning:", np.bincount(y_train))
print("Test dist after cleaning:", np.bincount(y_test))

Train dist after cleaning: [1579  135  139 1134  654  358]
Test dist after cleaning: [395  34  35 283 164  89]


In [24]:
from imblearn.over_sampling import SMOTE

min_class_size = np.bincount(y_train).min()
k = min(5, min_class_size - 1)
print(f"Using k_neighbors={k} (smallest class has {min_class_size} samples)")

smote = SMOTE(random_state=42, k_neighbors=k)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Train dist after SMOTE:", np.bincount(y_train_balanced))

Using k_neighbors=5 (smallest class has 135 samples)
Train dist after SMOTE: [1579 1579 1579 1579 1579 1579]


In [25]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

candidates = {
    "LDA": LinearDiscriminantAnalysis(),
    "SVC": SVC(C=1.0, kernel="rbf", class_weight="balanced"),
    "Ridge": RidgeClassifier(class_weight="balanced"),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        eval_metric="mlogloss"
    )
}

results = []
for name, model in candidates.items():
    model.fit(X_train_balanced, y_train_balanced)
    train_acc = accuracy_score(y_train_balanced, model.predict(X_train_balanced))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    gap = train_acc - test_acc
    results.append({"Model": name, "Train Acc": round(train_acc*100,2),
                     "Test Acc": round(test_acc*100,2), "Gap": round(gap*100,2)})
    print(f"\n=== {name} ===")
    print(classification_report(y_test, model.predict(X_test), target_names=le.classes_))

results_df = pd.DataFrame(results).sort_values("Test Acc", ascending=False)
print("\n=== SUMMARY ===")
print(results_df)


=== LDA ===
              precision    recall  f1-score   support

    Business       0.84      0.78      0.81       395
      Energy       0.60      0.94      0.74        34
      Health       0.78      0.89      0.83        35
     Markets       0.84      0.84      0.84       283
    Politics       0.84      0.84      0.84       164
  Technology       0.83      0.83      0.83        89

    accuracy                           0.82      1000
   macro avg       0.79      0.85      0.81      1000
weighted avg       0.83      0.82      0.82      1000


=== SVC ===
              precision    recall  f1-score   support

    Business       0.82      0.86      0.84       395
      Energy       0.79      0.76      0.78        34
      Health       0.94      0.86      0.90        35
     Markets       0.86      0.84      0.85       283
    Politics       0.89      0.87      0.88       164
  Technology       0.83      0.75      0.79        89

    accuracy                           0.84      10

In [26]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

candidates = {
    "LDA": LinearDiscriminantAnalysis(),
    "SVC": SVC(C=1.0, kernel="rbf", class_weight="balanced"),
    "Ridge": RidgeClassifier(class_weight="balanced"),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        eval_metric="mlogloss"
    )
}

results = []

for name, model in candidates.items():
    model.fit(X_train_balanced, y_train_balanced)

    train_pred = model.predict(X_train_balanced)
    test_pred = model.predict(X_test)

    train_acc_pct = accuracy_score(y_train_balanced, train_pred) * 100
    test_acc_pct = accuracy_score(y_test, test_pred) * 100
    gap_pct = train_acc_pct - test_acc_pct

    results.append({
        "Model": name,
        "Train Accuracy (%)": round(train_acc_pct, 2),
        "Test Accuracy (%)": round(test_acc_pct, 2),
        "Gap (%)": round(gap_pct, 2)
    })

    print(f"\n=== {name} ===")
    print(f"Train Accuracy: {train_acc_pct:.2f}%")
    print(f"Test Accuracy:  {test_acc_pct:.2f}%")
    print(f"Gap:            {gap_pct:.2f}%")
    print(classification_report(y_test, test_pred, target_names=le.classes_))

results_df = pd.DataFrame(results).sort_values("Test Accuracy (%)", ascending=False)
print("\n=== FINAL SUMMARY (%) ===")
print(results_df.to_string(index=False))


=== LDA ===
Train Accuracy: 94.71%
Test Accuracy:  82.20%
Gap:            12.51%
              precision    recall  f1-score   support

    Business       0.84      0.78      0.81       395
      Energy       0.60      0.94      0.74        34
      Health       0.78      0.89      0.83        35
     Markets       0.84      0.84      0.84       283
    Politics       0.84      0.84      0.84       164
  Technology       0.83      0.83      0.83        89

    accuracy                           0.82      1000
   macro avg       0.79      0.85      0.81      1000
weighted avg       0.83      0.82      0.82      1000


=== SVC ===
Train Accuracy: 98.97%
Test Accuracy:  84.50%
Gap:            14.47%
              precision    recall  f1-score   support

    Business       0.82      0.86      0.84       395
      Energy       0.79      0.76      0.78        34
      Health       0.94      0.86      0.90        35
     Markets       0.86      0.84      0.85       283
    Politics       0.8

#Trying to Resolve Overfitting

In [27]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Train SVC WITHOUT SMOTE, using class_weight instead — often more honest than synthetic oversampling
svc_no_smote = SVC(C=1.0, kernel="rbf", class_weight="balanced")
svc_no_smote.fit(X_train, y_train)  # X_train, y_train = pre-SMOTE, cleaned

train_acc = accuracy_score(y_train, svc_no_smote.predict(X_train)) * 100
test_acc = accuracy_score(y_test, svc_no_smote.predict(X_test)) * 100

print(f"SVC (no SMOTE, class_weight=balanced)")
print(f"Train: {train_acc:.2f}%  Test: {test_acc:.2f}%  Gap: {train_acc-test_acc:.2f}%")
print(classification_report(y_test, svc_no_smote.predict(X_test), target_names=le.classes_))

SVC (no SMOTE, class_weight=balanced)
Train: 94.25%  Test: 83.00%  Gap: 11.25%
              precision    recall  f1-score   support

    Business       0.85      0.80      0.82       395
      Energy       0.77      0.79      0.78        34
      Health       0.89      0.89      0.89        35
     Markets       0.82      0.83      0.82       283
    Politics       0.82      0.89      0.85       164
  Technology       0.82      0.84      0.83        89

    accuracy                           0.83      1000
   macro avg       0.83      0.84      0.83      1000
weighted avg       0.83      0.83      0.83      1000



In [28]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.1, 0.5, 1, 2, 5],
    "gamma": ["scale", 0.01, 0.1],
    "kernel": ["rbf"]
}

grid = GridSearchCV(
    SVC(class_weight="balanced"),
    param_grid,
    cv=3,
    scoring="balanced_accuracy",
    n_jobs=-1,
    verbose=1
)

# Use whichever training set performed better in Cell 9 — try WITHOUT SMOTE first
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV balanced accuracy:", grid.best_score_)

best_svc = grid.best_estimator_
train_acc = accuracy_score(y_train, best_svc.predict(X_train)) * 100
test_acc = accuracy_score(y_test, best_svc.predict(X_test)) * 100

print(f"\nTuned SVC — Train: {train_acc:.2f}%  Test: {test_acc:.2f}%  Gap: {train_acc-test_acc:.2f}%")
print(classification_report(y_test, best_svc.predict(X_test), target_names=le.classes_))

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best params: {'C': 2, 'gamma': 0.1, 'kernel': 'rbf'}
Best CV balanced accuracy: 0.8474787042959845

Tuned SVC — Train: 84.90%  Test: 79.60%  Gap: 5.30%
              precision    recall  f1-score   support

    Business       0.87      0.71      0.78       395
      Energy       0.52      0.94      0.67        34
      Health       0.72      0.94      0.81        35
     Markets       0.81      0.81      0.81       283
    Politics       0.80      0.86      0.83       164
  Technology       0.72      0.89      0.80        89

    accuracy                           0.80      1000
   macro avg       0.74      0.86      0.78      1000
weighted avg       0.81      0.80      0.80      1000



In [29]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Train SVC WITHOUT SMOTE, using class_weight instead — often more honest than synthetic oversampling
svc_no_smote = SVC(C=1.0, kernel="rbf", class_weight="balanced")
svc_no_smote.fit(X_train, y_train)  # X_train, y_train = pre-SMOTE, cleaned

train_acc = accuracy_score(y_train, svc_no_smote.predict(X_train)) * 100
test_acc = accuracy_score(y_test, svc_no_smote.predict(X_test)) * 100

print(f"SVC (no SMOTE, class_weight=balanced)")
print(f"Train: {train_acc:.2f}%  Test: {test_acc:.2f}%  Gap: {train_acc-test_acc:.2f}%")
print(classification_report(y_test, svc_no_smote.predict(X_test), target_names=le.classes_))

SVC (no SMOTE, class_weight=balanced)
Train: 94.25%  Test: 83.00%  Gap: 11.25%
              precision    recall  f1-score   support

    Business       0.85      0.80      0.82       395
      Energy       0.77      0.79      0.78        34
      Health       0.89      0.89      0.89        35
     Markets       0.82      0.83      0.82       283
    Politics       0.82      0.89      0.85       164
  Technology       0.82      0.84      0.83        89

    accuracy                           0.83      1000
   macro avg       0.83      0.84      0.83      1000
weighted avg       0.83      0.83      0.83      1000



In [30]:
param_grid_v2 = {
    "C": [1, 2, 3, 5, 10],
    "gamma": ["scale", 0.05, 0.1, 0.2],
    "kernel": ["rbf"]
}

grid_v2 = GridSearchCV(
    SVC(class_weight="balanced"),
    param_grid_v2,
    cv=3,
    scoring="balanced_accuracy",
    n_jobs=-1,
    verbose=1
)

grid_v2.fit(X_train, y_train)

print("Best params:", grid_v2.best_params_)
print("Best CV balanced accuracy:", grid_v2.best_score_)

best_svc_v2 = grid_v2.best_estimator_
train_acc = accuracy_score(y_train, best_svc_v2.predict(X_train)) * 100
test_acc = accuracy_score(y_test, best_svc_v2.predict(X_test)) * 100
print(f"\nTrain: {train_acc:.2f}%  Test: {test_acc:.2f}%  Gap: {train_acc-test_acc:.2f}%")
print(classification_report(y_test, best_svc_v2.predict(X_test), target_names=le.classes_))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'C': 3, 'gamma': 0.05, 'kernel': 'rbf'}
Best CV balanced accuracy: 0.8517256927060609

Train: 83.70%  Test: 79.00%  Gap: 4.70%
              precision    recall  f1-score   support

    Business       0.86      0.70      0.77       395
      Energy       0.51      0.94      0.66        34
      Health       0.70      0.94      0.80        35
     Markets       0.80      0.81      0.81       283
    Politics       0.81      0.84      0.82       164
  Technology       0.71      0.90      0.80        89

    accuracy                           0.79      1000
   macro avg       0.73      0.86      0.78      1000
weighted avg       0.81      0.79      0.79      1000



In [31]:
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression

svc_calibrated = CalibratedClassifierCV(
    SVC(C=best_svc_v2.C, gamma=best_svc_v2.gamma, kernel="rbf", class_weight="balanced"),
    cv=3
)

ensemble = VotingClassifier(
    estimators=[
        ("svc", svc_calibrated),
        ("lda", LinearDiscriminantAnalysis()),
        ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000))
    ],
    voting="soft",
    weights=[2, 1, 1]
)

ensemble.fit(X_train, y_train)

train_acc = accuracy_score(y_train, ensemble.predict(X_train)) * 100
test_acc = accuracy_score(y_test, ensemble.predict(X_test)) * 100
print(f"Ensemble — Train: {train_acc:.2f}%  Test: {test_acc:.2f}%  Gap: {train_acc-test_acc:.2f}%")
print(classification_report(y_test, ensemble.predict(X_test), target_names=le.classes_))

Ensemble — Train: 87.55%  Test: 82.20%  Gap: 5.35%
              precision    recall  f1-score   support

    Business       0.87      0.77      0.81       395
      Energy       0.67      0.94      0.78        34
      Health       0.84      0.89      0.86        35
     Markets       0.82      0.82      0.82       283
    Politics       0.82      0.90      0.85       164
  Technology       0.76      0.85      0.80        89

    accuracy                           0.82      1000
   macro avg       0.79      0.86      0.82      1000
weighted avg       0.83      0.82      0.82      1000



In [33]:
import joblib

# Retrain your best model (ensemble) on ALL cleaned data for the final deployable version
final_ensemble = VotingClassifier(
    estimators=[
        ("svc", CalibratedClassifierCV(SVC(C=best_svc_v2.C, gamma=best_svc_v2.gamma, kernel="rbf", class_weight="balanced"), cv=3)),
        ("lda", LinearDiscriminantAnalysis()),
        ("logreg", LogisticRegression(class_weight="balanced", max_iter=1000))
    ],
    voting="soft",
    weights=[2, 1, 1]
)

final_ensemble.fit(headline_embeddings, y_clean)  # train on ALL cleaned data

# Save model, encoder, and embedding model reference
joblib.dump(final_ensemble, "news_classifier_ensemble.pkl")
joblib.dump(le, "label_encoder.pkl")

# Save cleaned dataset
df.to_csv("News_Classification_Dataset_CLEANED.csv", index=False)

print("Saved: news_classifier_ensemble.pkl, label_encoder.pkl, News_Classification_Dataset_CLEANED.csv")

Saved: news_classifier_ensemble.pkl, label_encoder.pkl, News_Classification_Dataset_CLEANED.csv


In [34]:
import joblib
from sklearn.metrics import accuracy_score, classification_report

# Load what you saved
loaded_model = joblib.load("news_classifier_ensemble.pkl")
loaded_le = joblib.load("label_encoder.pkl")

# Check accuracy on your existing test set (X_test, y_test from before)
test_pred = loaded_model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred) * 100

print(f"Loaded model Test Accuracy: {test_acc:.2f}%")
print(classification_report(y_test, test_pred, target_names=loaded_le.classes_))

Loaded model Test Accuracy: 86.90%
              precision    recall  f1-score   support

    Business       0.90      0.81      0.85       395
      Energy       0.74      1.00      0.85        34
      Health       0.82      0.94      0.88        35
     Markets       0.84      0.87      0.86       283
    Politics       0.90      0.93      0.92       164
  Technology       0.85      0.94      0.89        89

    accuracy                           0.87      1000
   macro avg       0.84      0.92      0.87      1000
weighted avg       0.87      0.87      0.87      1000



In [35]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    final_ensemble,  # reuse the untrained pipeline structure, not the fitted one
    headline_embeddings, y_clean,
    cv=5, scoring="accuracy", n_jobs=-1
)

print(f"5-fold CV accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")
print(f"Individual folds: {[f'{s*100:.2f}%' for s in cv_scores]}")

5-fold CV accuracy: 83.34% (+/- 1.11%)
Individual folds: ['83.70%', '81.40%', '84.50%', '84.20%', '82.88%']
